# Treino em escala completa -- RSNA Knee Abnormality Detection (Kaggle Notebook)

Antes de rodar, configure o notebook (menu direito, "Notebook options"):
- **Accelerator**: GPU
- **Internet**: On (necessario para clonar o repositorio e baixar os pesos pre-treinados do backbone de imagem)
- **Add Data**: anexe a competicao `rsna-knee-abnormality-detection`

Este notebook clona o codigo de https://github.com/andreluizpedroso/rsna-knee-abnormality-detection,
instala as dependencias que faltam no ambiente padrao do Kaggle e roda `src.train`
no dataset completo (nao numa amostra pequena, como o `--smoke-test` local).

O checkpoint treinado e salvo em `/kaggle/working/checkpoints/` -- tudo dentro
de `/kaggle/working/` vira automaticamente o Output deste notebook, que pode
ser anexado como input a um notebook de submissao separado (inferencia,
sem internet).

## 1. Clonar o repositorio

In [ ]:
import os
import socket
import urllib.request
import zipfile

# git clone via `!` pode travar indefinidamente (em vez de falhar rapido) se
# a rede sandboxed do Kaggle nao repassar o trafego do jeito que o git
# espera. Baixar o .zip via urllib com timeout curto falha rapido em vez de
# ficar preso por horas sem log nenhum.
socket.setdefaulttimeout(30)

REPO_ZIP_URL = "https://github.com/andreluizpedroso/rsna-knee-abnormality-detection/archive/refs/heads/main.zip"
REPO_DIR = "/kaggle/working/rsna-knee-abnormality-detection"

if not os.path.exists(REPO_DIR):
    zip_path = "/kaggle/working/repo.zip"
    print("Baixando repositorio...")
    urllib.request.urlretrieve(REPO_ZIP_URL, zip_path)
    print("Extraindo...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/kaggle/working/")
    os.rename("/kaggle/working/rsna-knee-abnormality-detection-main", REPO_DIR)
    os.remove(zip_path)
    print("Pronto.")
else:
    print("Repositorio ja existe, pulando.")

%cd {REPO_DIR}

## 2. Instalar dependencias

`torch`/`torchvision`/`numpy`/`pandas`/`scikit-learn`/`opencv` ja vem
pre-instalados e configurados para a GPU do ambiente Kaggle -- reinstalar
por cima arrisca quebrar a versao compativel com CUDA. So instala o que
falta.

In [ ]:
!pip install -q pydicom timm iterative-stratification

## 3. Checagem do ambiente

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Diagnostico: lista o que realmente esta montado em /kaggle/input --
# necessario pra confirmar que os dados da competicao foram anexados (e com
# qual nome de pasta) antes de assumir o path em config.py.
print("Conteudo de /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "NAO EXISTE")

from src import config
print("IS_KAGGLE:", config.IS_KAGGLE)
print("DATA_DIR:", config.DATA_DIR)
print("train.csv encontrado:", config.TRAIN_CSV.exists())
print("train_series encontrado:", config.TRAIN_SERIES_DIR.exists())

assert config.TRAIN_CSV.exists(), (
    f"train.csv nao encontrado em {config.TRAIN_CSV} -- confira o conteudo de "
    f"/kaggle/input acima e se a competicao foi anexada em 'Add Data'."
)

## 4. Treino

Roda `src.train` via subprocess (evita o argparse do script brigar com os
argumentos internos do kernel do Jupyter). Sem `--smoke-test`: usa todos os
estudos com label real + pseudo-labels de weak supervision, conforme
`src/config.py` (`N_FOLDS=3`, treina o fold 0 -- ver comentario em
`src/train.py` sobre rodar os demais folds depois).

In [ ]:
import subprocess

# check=True (via subprocess, nao `!`) -- o `!comando` do Jupyter nao falha a
# celula sozinho quando o subprocesso retorna codigo != 0 (foi assim que o
# run anterior travou/quebrou no meio mas o notebook ainda reportou
# "COMPLETE" pro Kaggle). Isso garante que uma falha real do treino
# interrompe o notebook e fica visivel como erro no kernel.
subprocess.run([sys.executable, "-m", "src.train"], check=True)

## 5. Conferir o checkpoint salvo

In [ ]:
checkpoint_dir = "/kaggle/working/checkpoints"
print(os.listdir(checkpoint_dir) if os.path.exists(checkpoint_dir) else "Nenhum checkpoint encontrado -- ver logs do treino acima.")

## Proximo passo

Depois de rodar este notebook ("Save & Run All"), o Output (`/kaggle/working/checkpoints/`)
fica disponivel como um Kaggle Dataset anexavel. O notebook de submissao
(inferencia, sem internet) deve anexar:
- os dados da competicao,
- este Output (checkpoint treinado),
- os pesos do backbone de imagem (`timm`) salvos offline -- necessario porque
  o notebook de submissao nao tem internet para baixar do Hub.

O notebook de submissao ainda nao foi criado.